In [2]:
import sys
from pathlib import Path

ROOT = Path().resolve().parents[0]  # go up from notebooks/
sys.path.append(str(ROOT))

In [3]:
from config import SEED, N_LIST, METHODS, RESULTS_PATH
from models.cvae import train_cvae_on_arrays, sample_cvae_dataset
from models.bootstrap import sample_bootstrap
from models.gmm import sample_gmm
import numpy as np
import pandas as pd
from metrics import *
import ipywidgets as widgets
from IPython.display import display

# Loading Data
1. Breast cancer
2. Pima Indians Diabetes
3. Somethng else 2

#### 1. Breast Cancer

| Dataset | Category | n | p | Class 0 | Class 1 | Source | Notes |
|---------|----------|---|---|--------|--------|--------|-------|
| Breast Cancer | clinical_tabular | 569 | 30 | 212 | 357 | sklearn | moderate n, moderate p, numeric only |

#### 2. Diabetes

| Dataset | Category | n | p | Class 0 | Class 1 | Source | Notes |
|---------|----------|---|---|--------|--------|--------|-------|
| Diabetes| metabolic_tabular | 768 | 8 | 500 | 268 | openml | high n, low p, numeric only |

In [21]:
from loaders import load_breast, load_diabetes
data = load_diabetes()

X = data["X"]
y = data["y"]
feature_names = data["feature_names"]

print(data["dataset"], data["category"])
print(X.shape, y.shape)
print(np.bincount(y))

diabetes metabolic_tabular
(768, 8) (768,)
[500 268]


# Helper Functions
- Stratify Sampler

In [5]:
X_small, y_small, idx_small = stratified_subsample(X, y, n0=23, n1=68, seed=42)
print(X_small.shape)
print(np.bincount(y_small))

(91, 30)
[23 68]


- Train & sample CVAE

In [6]:

best_state = train_cvae_on_arrays(
    X,
    y,
    seed=42,
    z_dim=16,
    hidden=128,
    beta=0.5,
    lr=1e-3,
    epochs=200,
    batch_size=32,
)

X_syn, y_syn = sample_cvae_dataset(
    best_state,
    n0=23,
    n1=68,
    seed=42
)

print(X_syn.shape)
print(np.bincount(y_syn))

Epoch  50 | val loss=8.4903 recon=5.1531 kl=6.6744
Epoch 100 | val loss=7.9788 recon=4.6182 kl=6.7211
Epoch 150 | val loss=8.2599 recon=5.0080 kl=6.5038
Epoch 200 | val loss=7.7110 recon=4.5055 kl=6.4109
(91, 30)
[23 68]


- bootstrap

In [ ]:
X_syn_b, y_syn_b = sample_bootstrap(X_small, y_small, n0=23, n1=68, seed=42)

- gmm

In [ ]:
X_syn_g, y_syn_g = sample_gmm(X_small, y_small, n0=23, n1=68, seed=42, n_components=2)

# The actual tests with numbers or something

In [ ]:
metrics = evaluate_all(X_real=X, X_syn=X_syn, y_real= y, y_syn = y_syn)
print(metrics)

{'rf_auc_mean': np.float64(0.5924777777777778), 'rf_auc_sd': np.float64(0.10311534414203768), 'rf_sep_mean': np.float64(0.6179444444444445), 'rf_sep_sd': np.float64(0.07262245936993783), 'corr_mean_abs_diff': np.float64(0.098457593515556), 'corr_max_abs_diff': np.float64(0.31390714081717935), 'pval_mean': np.float64(0.5916932475002658), 'pval_median': np.float64(0.5108325453308513), 'prop_significant': np.float64(0.0)}


### Results row

In [14]:
datasets = [
    load_breast,
    load_diabetes,
]
results = []

for method, X_syn, y_syn in [
    ("bootstrap", X_syn_b, y_syn_b),
    ("gmm", X_syn_g, y_syn_g),
    ("cvae", X_syn, y_syn),
]:
    # metrics = evaluate_all(X, y, X_syn, y_syn)

    row = {
        "dataset": data["dataset"],
        "category": data["category"],
        "subgroup": "full",
        "n0": int((y_small == 0).sum()),
        "n1": int((y_small == 1).sum()),
        "n_total": len(y_small),
        "method": method,
        "seed": 42,
        "rf_auc": metrics["rf_auc_mean"],
        "corr_mean_abs_diff": metrics["corr_mean_abs_diff"],
        "prop_significant": metrics["prop_significant"],
    }

    results.append(row)

df = pd.DataFrame(results)
display(df)

,dataset,category,subgroup,n0,n1,n_total,method,seed,rf_auc,corr_mean_abs_diff,prop_significant
0,breast_cancer,clinical_tabular,full,23,68,91,bootstrap,42,0.592478,0.098458,0.0
1,breast_cancer,clinical_tabular,full,23,68,91,gmm,42,0.592478,0.098458,0.0
2,breast_cancer,clinical_tabular,full,23,68,91,cvae,42,0.592478,0.098458,0.0


In [22]:
import pandas as pd
import ipywidgets as widgets
from IPython.display import display

def run_experiment(n0, n1):
    datasets = [
        load_breast,
        load_diabetes,
    ]

    methods = ["bootstrap", "gmm", "cvae"]
    rows = []

    for load_fn in datasets:
        data = load_fn()
        X = data["X"]
        y = data["y"]

        X_small, y_small, _ = stratified_subsample(
            X, y,
            n0=n0,
            n1=n1,
            seed=42
        )

        for method in methods:
            if method == "bootstrap":
                X_syn, y_syn = sample_bootstrap(X_small, y_small, n0, n1, seed=42)

            elif method == "gmm":
                X_syn, y_syn = sample_gmm(X_small, y_small, n0, n1, seed=42)

            elif method == "cvae":
                print("Training CVAE for", data["dataset"])
                best = train_cvae_on_arrays(
                    X_small,
                    y_small,
                    seed=42,
                    epochs=100,   
                    batch_size=32,

                )
                X_syn, y_syn = sample_cvae_dataset(best, n0, n1, seed=42)

            metrics = evaluate_all(X_real=X, y_real= y,  X_syn=X_syn, y_syn = y_syn)

            rows.append({
                "dataset": data["dataset"],
                "category": data["category"],
                "n0": n0,
                "n1": n1,
                # "n_total": n0 + n1,
                "method": method,
                "seed": 42,
                "rf_auc": metrics["rf_auc_mean"],
                "corr_mean_abs_diff": metrics["corr_mean_abs_diff"],
                "prop_significant": metrics["prop_significant"],
            })

    df = pd.DataFrame(rows)
    display(df)

In [23]:
n0_widget = widgets.IntSlider(min=5, max=50, step=1, value=23, description="n0")
n1_widget = widgets.IntSlider(min=10, max=100, step=1, value=68, description="n1")
button = widgets.Button(description="Run")
output = widgets.Output()

def on_click(b):
    with output:
        output.clear_output()
        run_experiment(n0_widget.value, n1_widget.value)

button.on_click(on_click)

display(n0_widget, n1_widget, button, output)

IntSlider(value=23, description='n0', max=50, min=5)

IntSlider(value=68, description='n1', min=10)

Button(description='Run', style=ButtonStyle())

Output()